# Duckietown Explore Retrain (Stable Path)

Uses the existing stable trainer: `/content/leworldduckie/src/train.py`.


In [ ]:
# Config — exploratory retrain, planned fs6 config (see docs/memory + T6 gate)
REPO_URL = 'https://github.com/giuliovv/leworldduckie.git'
DATA_URL = 'https://leworldduckie-public-data.s3.us-east-1.amazonaws.com/duckie_explore.h5'
DATA_LOCAL = '/content/duckie_explore.h5'
RUN_ID = 'duckie_explore_fs6'
# Artifact destination — the leworldduckie-colab key is scoped to this bucket
# (ERP AWS account). Synced back to s3://leworldduckie/training/runs/ afterwards.
ARTIFACT_BUCKET = 'leworldduckie-colab-artifacts'
EPOCHS = 50
BATCH_SIZE = 128

# Model/training config — MUST stay in sync with the eval side:
# t6_eval.py and mpc_controller.py read the same ACTION_SCALE env var.
TRAIN_ENV = {
    'FRAMESKIP':    '6',
    'N_PREDS':      '4',
    'SIGREG_W':     '0.1',
    'ACTION_SCALE': '5.3',
}


In [ ]:
# AWS credentials — OPTIONAL. With them, checkpoints auto-upload to S3 as
# training progresses (safest — survives a Colab disconnect). Without them,
# training still runs fine; you manually download /tmp/run_<RUN_ID>/checkpoint_best.pt
# from the Colab file browser at the end and upload it yourself (the old flow).
#
# To use keys: add Colab secrets AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY
# (key icon, left sidebar) BEFORE running this cell, and toggle notebook
# access on for this notebook.
import os

UPLOAD_TO_S3 = True  # set False to skip credentials entirely and download manually

if UPLOAD_TO_S3:
    try:
        from google.colab import userdata
        os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
        os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
        os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
        import subprocess
        subprocess.check_call(['bash', '-lc', 'pip -q install boto3'])
        import boto3
        ident = boto3.client('sts').get_caller_identity()
        print('S3 upload identity OK:', ident['Arn'])
    except Exception as e:
        print(f'No usable AWS credentials ({e}) — continuing WITHOUT S3 upload.')
        print('You will need to manually download the checkpoint at the end.')
        UPLOAD_TO_S3 = False
else:
    print('S3 upload disabled by choice — manual download required at the end.')


In [ ]:
# Setup
import os, subprocess
subprocess.check_call(['bash','-lc','python3 -m pip install -U pip'])
# Matches infra/launch_training.sh's install list exactly — stable-worldmodel[train]
# provides stable_pretraining, which builds the real ViT-Tiny encoder. Without it,
# train.py silently falls back to a throwaway CNN encoder (see ALLOW_CNN_FALLBACK
# in src/train.py) and the whole run is architecturally invalid.
subprocess.check_call(['bash','-lc',
    'pip install -q torch torchvision einops h5py matplotlib scikit-learn boto3 "stable-worldmodel[train]"'])
subprocess.check_call(['bash','-lc', f'git clone --depth 1 {REPO_URL} /content/leworldduckie || true'])
if not os.path.exists(DATA_LOCAL):
    subprocess.check_call(['bash','-lc', f'wget -O {DATA_LOCAL} "{DATA_URL}"'])
print('data:', DATA_LOCAL, os.path.getsize(DATA_LOCAL))

# Verify the real encoder is actually importable BEFORE spending GPU time training —
# this is exactly the check that was missing when the CNN-fallback run happened.
import importlib
importlib.import_module('stable_pretraining')
print('stable_pretraining import OK — ViT-Tiny encoder will be used')


In [ ]:
# Train (stable src/train.py path)
import os, subprocess
env = os.environ.copy()
env['DATA_PATH'] = DATA_LOCAL
env['S3_DATA_KEY'] = 'data/duckie_explore.h5'
env['S3_BUCKET'] = ARTIFACT_BUCKET
env['S3_UPLOAD_ENABLED'] = 'true' if UPLOAD_TO_S3 else 'false'
env['N_EPOCHS'] = str(EPOCHS)
env['BATCH_SIZE'] = str(BATCH_SIZE)
env.update(TRAIN_ENV)
cmd = f'cd /content/leworldduckie && python3 src/train.py --run-id {RUN_ID} --epochs {EPOCHS} 2>&1 | tee /content/train_duckie.log'
print(cmd)
print('env:', {k: env[k] for k in ('FRAMESKIP','N_PREDS','SIGREG_W','ACTION_SCALE','S3_UPLOAD_ENABLED')})
ret = subprocess.run(['bash','-lc',cmd], env=env)
if ret.returncode != 0:
    raise RuntimeError(f'training failed: {ret.returncode}; see /content/train_duckie.log')


In [ ]:
# Verify the checkpoint is somewhere safe.
# NOTE: train.py's local file is always named checkpoint_latest.pt —
# "checkpoint_best.pt" is only an S3 key alias it uploads the same file
# under when that epoch happens to be the best so far. There is no local
# file literally named checkpoint_best.pt.
import glob, os

local_ckpt = f'/tmp/run_{RUN_ID}/checkpoint_latest.pt'

if UPLOAD_TO_S3:
    import boto3
    s3 = boto3.client('s3')
    prefix = f'training/runs/{RUN_ID}/'
    resp = s3.list_objects_v2(Bucket=ARTIFACT_BUCKET, Prefix=prefix)
    keys = [o['Key'] for o in resp.get('Contents', [])]
    print('\n'.join(keys) or '(nothing)')
    assert any(k.endswith('checkpoint_best.pt') for k in keys), (
        f'checkpoint_best.pt NOT on S3 under {prefix} — '
        'copy /tmp/run_*/checkpoint_latest.pt out of this runtime NOW before it is recycled')
    print(f'\nOK: s3://{ARTIFACT_BUCKET}/{prefix}checkpoint_best.pt')
else:
    matches = glob.glob(local_ckpt)
    if not matches:
        print(f'{local_ckpt} not found — log tail:')
        print(open('/content/train_duckie.log').read()[-3000:])
        raise AssertionError(f'{local_ckpt} not found — see log above')
    size_mb = os.path.getsize(local_ckpt) / 1e6
    print(f'Training finished, no S3 upload was configured.')
    print(f'Checkpoint is at: {local_ckpt} ({size_mb:.0f} MB)')
    print('Download it now (Colab file browser, left sidebar → find the path → download)')
    print('before this runtime disconnects, then send it to Claude for the T6 eval.')
